In [1]:
import os, warnings
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
warnings.filterwarnings("ignore")
FILE="Water Quality.csv"
TARGET_COLUMN=None   # Set only to a real, validated target column
TASK_TYPE=None       # "classification" or "regression" only if target exists
def locate(name):
    if os.path.isfile(name): return os.path.abspath(name)
    for base in [os.getcwd(),os.path.expanduser("~/Desktop"),os.path.expanduser("~/Downloads"),os.path.expanduser("~/Documents")]:
        if os.path.isdir(base):
            for root,dirs,files in os.walk(base):
                dirs[:]=[d for d in dirs if d not in {".git","node_modules",".venv","venv","__pycache__"}]
                for f in files:
                    if f.lower()==name.lower(): return os.path.join(root,f)
    from tkinter import Tk,filedialog
    r=Tk(); r.withdraw(); r.attributes("-topmost",True)
    p=filedialog.askopenfilename(title="Select Water Quality CSV",filetypes=[("CSV files","*.csv")]); r.destroy()
    return p or None
path=locate(FILE)
if not path: raise FileNotFoundError("Select Water Quality.csv or update FILE to its full path.")
df=pd.read_csv(path,low_memory=False); df.columns=df.columns.str.strip()
req={"MonitoringLocationID","ActivityStartDate","CharacteristicName","ResultValue"}
if not req.issubset(df.columns): raise ValueError(f"Missing columns: {sorted(req-set(df.columns))}")
df["ActivityStartDate"]=pd.to_datetime(df["ActivityStartDate"],errors="coerce")
df["ResultValue"]=pd.to_numeric(df["ResultValue"],errors="coerce")
df=df.dropna(subset=["MonitoringLocationID","ActivityStartDate","CharacteristicName","ResultValue"]).copy()
df["YearMonth"]=df["ActivityStartDate"].dt.to_period("M").astype(str)
m=df.groupby(["MonitoringLocationID","YearMonth","CharacteristicName"])["ResultValue"].median().reset_index()
wide=m.pivot_table(index=["MonitoringLocationID","YearMonth"],columns="CharacteristicName",values="ResultValue").reset_index()
wide.columns.name=None; ids=["MonitoringLocationID","YearMonth"]
raw=wide.drop(columns=ids).apply(pd.to_numeric,errors="coerce")
raw=raw.loc[:,[c for c in raw if raw[c].notna().any() and raw[c].nunique(dropna=True)>1]]
if raw.shape[1]<2 or len(wide)<3: raise ValueError("Need at least 3 records and 2 varying numeric parameters.")
X=StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(raw))
out=os.path.join(os.path.dirname(path),"task1_clustering_outputs"); os.makedirs(out,exist_ok=True)
wide.to_csv(os.path.join(out,"monthly_feature_table.csv"),index=False)
metrics=[]; k=min(3,len(wide)-1)
km=KMeans(n_clusters=k,n_init=10,random_state=42); kl=km.fit_predict(X)
gm=GaussianMixture(n_components=k,random_state=42,reg_covar=1e-6); gl=gm.fit_predict(X); probs=gm.predict_proba(X)
db=DBSCAN(eps=1.5,min_samples=max(3,min(10,int(np.sqrt(len(wide)))))); dl=db.fit_predict(X)
def sil(x,l):
    return silhouette_score(x,l) if 1<len(set(l))<len(l) else np.nan
for name,labels in [("KMeans",kl),("GMM",gl),("DBSCAN",dl)]:
    mask=(labels!=-1) if name=="DBSCAN" else np.ones(len(labels),dtype=bool)
    metrics.append({"Algorithm":name,"Clusters_including_noise":len(set(labels)),"Silhouette":sil(X[mask],labels[mask])})
feat=wide[ids].copy(); feat["KMeans_Cluster"]=kl; feat["DBSCAN_Cluster"]=dl; feat["GMM_Cluster"]=gl
for j in range(probs.shape[1]): feat[f"GMM_Probability_{j}"]=probs[:,j]
for j in range(km.transform(X).shape[1]): feat[f"KMeans_CentroidDistance_{j}"]=km.transform(X)[:,j]
feat.to_csv(os.path.join(out,"cluster_based_features.csv"),index=False)
pd.DataFrame(metrics).to_csv(os.path.join(out,"clustering_metrics.csv"),index=False)
# Optional simple holdout evaluation; not a substitute for grouped/time-aware validation.
if TARGET_COLUMN and TARGET_COLUMN in wide and TASK_TYPE in ("classification","regression"):
    y=wide[TARGET_COLUMN]; valid=y.notna(); y=y[valid]; Xb=X[valid.to_numpy()]
    cf=feat.drop(columns=ids).apply(pd.to_numeric,errors="coerce")
    Xc=SimpleImputer(strategy="median").fit_transform(cf.loc[valid])
    Xa=np.column_stack([Xb,Xc])
    if len(y)>=10 and (TASK_TYPE=="regression" or y.nunique()>=2):
        strat=y if TASK_TYPE=="classification" and y.value_counts().min()>=2 else None
        indices=np.arange(len(y)); tr,te=train_test_split(indices,test_size=.25,random_state=42,stratify=strat)
        Model=RandomForestClassifier if TASK_TYPE=="classification" else RandomForestRegressor
        a=Model(n_estimators=250,random_state=42).fit(Xb[tr],y.iloc[tr])
        b=Model(n_estimators=250,random_state=42).fit(Xa[tr],y.iloc[tr])
        p1,p2=a.predict(Xb[te]),b.predict(Xa[te])
        if TASK_TYPE=="classification":
            ev=pd.DataFrame([{"Model":"Parameters only","Accuracy":accuracy_score(y.iloc[te],p1),"F1_weighted":f1_score(y.iloc[te],p1,average="weighted",zero_division=0)},{"Model":"Parameters + cluster features","Accuracy":accuracy_score(y.iloc[te],p2),"F1_weighted":f1_score(y.iloc[te],p2,average="weighted",zero_division=0)}])
        else:
            ev=pd.DataFrame([{"Model":"Parameters only","MAE":mean_absolute_error(y.iloc[te],p1),"RMSE":mean_squared_error(y.iloc[te],p1)**.5,"R2":r2_score(y.iloc[te],p1)},{"Model":"Parameters + cluster features","MAE":mean_absolute_error(y.iloc[te],p2),"RMSE":mean_squared_error(y.iloc[te],p2)**.5,"R2":r2_score(y.iloc[te],p2)}])
        ev.to_csv(os.path.join(out,"predictive_comparison.csv"),index=False)
else: print("No validated target configured: supervised predictive evaluation skipped.")
print("Task 1 outputs:",out)
print("Clusters are exploratory patterns, not safe/unsafe labels.")


No validated target configured: supervised predictive evaluation skipped.
Task 1 outputs: c:\Users\YASH\Desktop\BrainyBeam\task3 web\task1_clustering_outputs
Clusters are exploratory patterns, not safe/unsafe labels.
